# 02 — Transformasi ETL dengan PySpark (Google Colab)

Notebook ini menjalankan alur **Extract → Transform → Load** pada data transaksi penuh. Jalankan sel secara berurutan. Proses pertama dapat memakan waktu beberapa menit karena Spark membaca hampir 10 juta baris.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'pyspark==3.5.5', 'pyarrow>=15,<21', 'python-dotenv>=1.0,<2.0',
    'psycopg2-binary>=2.9,<3.0', 'SQLAlchemy>=2.0,<3.0',
])

try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError('Notebook ini khusus Google Colab. Gunakan versi notebooks/local untuk Jupyter.') from exc

drive.mount('/content/drive')

REPOSITORY_URL = 'https://github.com/USERNAME/big-data-retail-analytics.git'
PROJECT_ROOT = Path('/content/big-data-retail-analytics')
if not (PROJECT_ROOT / 'src').is_dir():
    if 'USERNAME' in REPOSITORY_URL:
        raise ValueError('Ganti USERNAME pada REPOSITORY_URL dengan username GitHub milikmu.')
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(PROJECT_ROOT)], check=True)

COLAB_DATASET_ROOT = '/content/drive/MyDrive/Synthetic Retail Dataset - 10 Million Transactions'
DATASET_ROOT = Path(COLAB_DATASET_ROOT)
if not (DATASET_ROOT / 'retail_contaminated_dataset').is_dir():
    raise FileNotFoundError(f'Folder dataset tidak ditemukan: {DATASET_ROOT}')

OUTPUT_ROOT = Path('/content/drive/MyDrive/big-data-retail-analytics-output')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
os.environ['DATASET_ROOT'] = str(DATASET_ROOT)
os.environ['OUTPUT_DIR'] = str(OUTPUT_ROOT)
os.environ['PYSPARK_SUBMIT_ARGS'] = '--driver-memory 4g pyspark-shell'
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
from pyspark import StorageLevel
from src.config import get_paths, validate_source_files
from src.spark_pipeline import (
    aggregate_outputs, create_spark, read_sources, transform_sources,
    validate_sources, write_csv_folder,
)

VARIANT = 'retail_contaminated_dataset'
paths = get_paths(VARIANT)
validate_source_files(paths)
paths

## 1. Extract
Spark membaca CSV memakai skema yang sudah ditentukan. Ini lebih aman dan lebih cepat daripada menebak tipe kolom setiap kali.

In [ ]:
spark = create_spark('local[2]')
frames = read_sources(spark, paths.source_dir)
{name: frame.columns for name, frame in frames.items()}

## 2. Validasi sumber
Pemeriksaan ini mencari nilai wajib yang kosong, angka tidak wajar, total transaksi yang tidak cocok, serta ID yang tidak mempunyai pasangan pada tabel master.

In [ ]:
checks = validate_sources(frames)
checks

## 3. Transform
Transaksi digabungkan dengan data toko, produk, pelanggan, dan promosi. Kolom analisis seperti bulan, penjualan kotor, diskon, laba kotor, margin, dan penanda promosi lalu dibuat.

In [ ]:
enriched = transform_sources(frames).persist(StorageLevel.DISK_ONLY)
output_rows = enriched.count()
assert output_rows == checks['input_rows'], 'Jumlah baris berubah setelah transformasi'
assert all(value == 0 for key, value in checks.items() if key != 'input_rows'), checks
print(f'Baris hasil: {output_rows:,}')
enriched.select(
    'date', 'receipt_id', 'store_name', 'sku_name', 'category',
    'quantity', 'total_value', 'gross_profit', 'channel', 'is_promoted'
).show(10, truncate=False)

In [ ]:
aggregates = aggregate_outputs(enriched)
aggregates['monthly_sales'].show(12, truncate=False)
aggregates['store_sales'].show(10, truncate=False)
aggregates['category_sales'].show(10, truncate=False)
aggregates['channel_sales'].show(truncate=False)
aggregates['promotion_comparison'].show(truncate=False)

## 4. Load
Hasil rinci disimpan sebagai Parquet yang dipartisi per tahun dan bulan. Ringkasan kecil disimpan sebagai CSV agar mudah dibuka atau dimuat ke PostgreSQL.

In [ ]:
paths.parquet_dir.parent.mkdir(parents=True, exist_ok=True)
paths.validation_dir.mkdir(parents=True, exist_ok=True)

enriched.write.mode('overwrite').partitionBy('year', 'month_key').parquet(
    str(paths.parquet_dir)
)
for name, frame in aggregates.items():
    write_csv_folder(frame, paths.validation_dir / name)

print('Parquet:', paths.parquet_dir)
print('Ringkasan CSV:', paths.validation_dir)

## 5. Verifikasi hasil load
Data dibaca ulang dari Parquet untuk memastikan jumlah baris tetap sama dan format hasil benar-benar dapat digunakan.

In [ ]:
reloaded = spark.read.parquet(str(paths.parquet_dir))
reloaded_rows = reloaded.count()
assert reloaded_rows == output_rows
print(f'Verifikasi berhasil: {reloaded_rows:,} baris')
reloaded.printSchema()

In [ ]:
enriched.unpersist()
spark.stop()